# Lab 14: Three routes to the same answer

**How well do different numerical methods recover a result we already know?**

We'll compare the **exact harmonic-oscillator solution**, a **finite-difference Schrödinger solver**, and **path-integral MCMC**.

Every method will give us the position density and its width, $\langle x^2\rangle$. The exact solution lets us separate numerical bias from sampling noise.

PHYS 142/242 · Friday, November 13, 2026

## What we are comparing

| Route | What we calculate | What we check |
|---|---|---|
| Exact solution | An analytic density and $\langle x^2\rangle=0.5$ | The common benchmark |
| Finite differences | The lowest eigenvector of a spatial Hamiltonian | Spatial grid and boundary error |
| Path-integral MCMC | A distribution of imaginary-time paths | Time grid, equilibration, sampling error |

First solve the eigenproblem. Then build the sampler and watch its data accumulate. Finally put both numerical answers on the **same plots**.

Here $\Delta x$ is a spatial grid spacing; $\Delta\tau$ is an imaginary-time step. A Monte Carlo sweep is neither of these.

## Set up the notebook

Run the cells in order. Each simulation below starts a new chain from a stated seed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh_tridiagonal
from numba import njit

plt.rcParams.update({
    "font.size": 12, "figure.figsize": (6.2, 3.6),
    "figure.dpi": 110, "figure.constrained_layout.use": True,
})
M, OMEGA, HBAR = 1.0, 1.0, 1.0
TAU = 30.0

We use $m=\omega=\hbar=1$. SciPy solves the deterministic eigenproblem; NumPy and Numba generate the random paths. Run all cells in order.

## A test case with an exact answer

For $V(x)=x^2/2$, the ground-state solution is

$$E_0=\frac12,\qquad \rho_0(x)=|\psi_0(x)|^2=\frac{e^{-x^2}}{\sqrt\pi},
\qquad \langle x^2\rangle_0=\int x^2\rho_0(x)\,dx=\frac12.$$

The particle is concentrated near the potential minimum, with a nonzero width.

In [ ]:
x_grid = np.linspace(-3, 3, 301)
rho_ground = np.exp(-x_grid**2) / np.sqrt(np.pi)
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(6.2, 4.0))
axes[0].plot(x_grid, 0.5 * x_grid**2, color="#a54632")
axes[0].axhline(0.5, color="black", ls="--", label="Ground energy")
axes[0].set(ylabel="Potential V(x)")
axes[0].legend(fontsize=10)
axes[1].plot(x_grid, rho_ground, color="#007c91", lw=2)
axes[1].fill_between(x_grid, rho_ground, alpha=0.2, color="#007c91")
axes[1].set(xlabel="Position x", ylabel="Density ρ(x)")
plt.show()

Both numerical methods must recover this same density and width. A visually plausible curve is only the first check.

## Method 1: Solve the Schrödinger equation

Approximate the second derivative on a spatial grid:

$$-\frac{\psi_{i+1}-2\psi_i+\psi_{i-1}}{2\Delta x^2}
+\frac{x_i^2}{2}\psi_i=E\psi_i.$$

This gives a symmetric tridiagonal matrix. Its **lowest eigenvector** approximates the ground-state wavefunction; squaring it gives the density.

Use $\psi(-5)=\psi(5)=0$. These artificial boundaries are far into the Gaussian tails. Unlike MCMC, this calculation has no random sampling error.

## Solve and normalize

The eigensolver returns a vector with sum of squares one. Divide by the square root of the spacing so the density integrates to one.

In [ ]:
def finite_difference_ground(n, extent=5.0):
    dx = 2 * extent / (n + 1)
    x = np.linspace(-extent + dx, extent - dx, n)
    diagonal = 1 / dx**2 + 0.5 * x**2
    off_diagonal = np.full(n - 1, -0.5 / dx**2)
    energy, vectors = eigh_tridiagonal(
        diagonal, off_diagonal, select="i", select_range=(0, 0))
    rho = vectors[:, 0]**2 / dx
    x2 = np.sum(x**2 * rho) * dx
    return x, rho, energy[0], x2, dx

fd_results = [finite_difference_ground(n) for n in (24, 49, 99, 199)]
for _, _, energy, x2, dx in fd_results:
    print(f"Δx={dx:.2f}: E₀={energy:.6f}, <x²>={x2:.6f}")

Normalization is $\sum_i\rho_i\Delta x=1$. The ground-state energy is a useful extra check; our shared comparison with MCMC will use density and $\langle x^2\rangle$.

## The deterministic density

The coarse and fine grids already look close to the exact curve. Can we tell how close from this plot alone?

In [ ]:
fig, ax = plt.subplots()
ax.plot(x_grid, rho_ground, color="black", lw=2, label="Exact")
for result, color in zip((fd_results[0], fd_results[-1]),
                         ("#a54632", "#007c91")):
    x, rho, _, _, dx = result
    ax.plot(x, rho, ".--", ms=3, color=color, label=f"FD Δx={dx:.2f}")
ax.set(xlim=(-3, 3), xlabel="Position x", ylabel="Density ρ(x)")
ax.legend()
plt.show()

The curves overlap strongly. Plotting their difference makes the numerical error visible.

## Subtract the known answer

Evaluate the exact density at each numerical grid point before subtracting.

In [ ]:
fig, ax = plt.subplots()
for result in fd_results[::2]:
    x, rho, _, _, dx = result
    exact_here = np.exp(-x**2) / np.sqrt(np.pi)
    ax.plot(x, rho - exact_here, label=f"FD Δx={dx:.2f}")
ax.axhline(0, color="black", lw=1)
ax.set(xlim=(-3, 3), xlabel="Position x", ylabel="FD density − exact density")
ax.legend()
plt.show()

The coarse grid makes the density too narrow: extra weight near the center and too little in the shoulders. Refining the grid reduces that bias.

## Does the spatial error shrink as expected?

The central-difference approximation has leading error proportional to $\Delta x^2$.

In [ ]:
fd_dx = np.array([r[4] for r in fd_results])
fd_x2 = np.array([r[3] for r in fd_results])
fd_error = np.abs(fd_x2 - 0.5)
fig, ax = plt.subplots()
ax.loglog(fd_dx, fd_error, "o-", color="#007c91", label="Width error")
ax.loglog(fd_dx, fd_error[0] * (fd_dx / fd_dx[0])**2,
          "--", color="#a54632", label="Reference slope: Δx²")
ax.set(xlabel="Spatial grid spacing Δx", ylabel="Absolute error in <x²>")
ax.set_xticks(fd_dx, [f"{dx:g}" for dx in fd_dx])
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.legend()
plt.show()

This is systematic error: rerunning the same matrix gives the same answer. Decreasing the spacing improves it; drawing more random samples is irrelevant to this solver.

## Method 2: Sample paths instead of eigenvectors

In Lab 11, one configuration was a whole spin lattice. Here it is a whole path:

$$x=(x_0,x_1,\ldots,x_{N_\tau-1}),\qquad \tau_i=i\,\Delta\tau.$$

The $x_i$ are **continuous positions**. Only imaginary time is on a grid. We sample paths with the quantum statistical weight, then pool their positions to estimate the same density $\rho(x)$.

One Monte Carlo update changes a coordinate. It does not advance physical time.

## Close the path

We impose $x_{N_\tau}=x_0$. The first and last stored points are neighbors.

In [ ]:
dt = 1.0
n_tau = int(round(TAU / dt))
tau_grid = np.arange(n_tau + 1) * dt
example_path = 0.8 * np.sin(2 * np.pi * np.arange(n_tau) / n_tau)
fig, ax = plt.subplots()
ax.plot(tau_grid, np.r_[example_path, example_path[0]], "o-", ms=3)
ax.set(xlabel=r"Imaginary time $\tau$", ylabel="Position x")
plt.show()

The final point repeats the first for drawing only. The array stores $N_\tau$ coordinates, with no duplicate endpoint.

## When can we compare with the ground state?

Periodic paths at finite total imaginary time sample a **thermal** distribution. For this oscillator,

$$\langle x^2\rangle_{\rm thermal}=\frac12\coth(\tau_{\rm total}/2).$$

In [ ]:
total_times = np.geomspace(0.3, 40, 200)
thermal_x2 = 0.5 / np.tanh(total_times / 2)
fig, ax = plt.subplots()
ax.semilogx(total_times, thermal_x2, color="#007c91", label="Exact thermal")
ax.axhline(0.5, color="black", ls="--", label="Ground state")
ax.plot(TAU, 0.5 / np.tanh(TAU / 2), "o", color="#a54632", label="Our total time: 30")
ax.set(xlabel="Total imaginary time", ylabel="Exact <x²>")
ax.legend(fontsize=10)
plt.show()

At total time 30, the thermal correction is below $10^{-12}$. The finite time **step** still introduces error; we will check that separately.

## Assign a weight to each path

$$P(x)\propto e^{-S_E(x)/\hbar},$$

$$S_E(x)=\sum_{i=0}^{N_\tau-1}\left[
\frac{m(x_{i+1}-x_i)^2}{2\Delta\tau}
+\Delta\tau\,\frac{m\omega^2 x_i^2}{2}\right].$$

The first term penalizes jumps between neighboring times. The second penalizes large positions. We evaluate the potential at the **left endpoint**, as in Assignment 3 and Javier's corrected lab.

## One link, then the whole path

Keep the action in small functions so we can check each piece.

In [ ]:
@njit
def action(x_left, x_right, dt):
    kinetic = 0.5 * M * (x_right - x_left)**2 / dt
    potential = 0.5 * M * OMEGA**2 * x_left**2 * dt
    return kinetic + potential

@njit
def total_action(path, dt):
    result = 0.0
    for i in range(path.size):
        result += action(path[i], path[(i + 1) % path.size], dt)
    return result

The modulo operation closes the last link. Each link and each potential term are counted once.

## What makes a path costly?

Compare the two pieces of the action for our example path and a proposal that adds one sharp bend.

In [ ]:
kinetic = 0.5 * (np.roll(example_path, -1) - example_path)**2 / dt
potential = 0.5 * example_path**2 * dt
spiked = example_path.copy()
spiked[n_tau // 2] += 0.65
spike_kinetic = 0.5 * (np.roll(spiked, -1) - spiked)**2 / dt
fig, ax = plt.subplots()
ax.plot(np.arange(n_tau), potential, label="Potential: original path")
ax.plot(np.arange(n_tau), kinetic, label="Kinetic: original path")
ax.plot(np.arange(n_tau), spike_kinetic, "--", label="Kinetic: one bend added")
ax.set(xlabel="Link index i", ylabel="Contribution to action", ylim=(0, 0.52))
ax.legend(fontsize=9, loc="upper center")
plt.show()

Large positions cost potential action. A sharp bend costs kinetic action on both neighboring links; the local update must include both.

## Change one point

Propose $x_i'=x_i+u$, with $u$ uniform on $[-h,h]$.

Only the two links touching $i$ change. We can compute $\Delta S_E$ without summing the entire path.

In [ ]:
i = n_tau // 2
proposal = example_path.copy()
proposal[i] += 0.65
fig, ax = plt.subplots()
ax.plot(tau_grid, np.r_[example_path, example_path[0]], "o-", label="Current")
ax.plot(tau_grid, np.r_[proposal, proposal[0]], "o--", label="Proposed")
ax.set(xlim=((i - 3) * dt, (i + 3) * dt),
       xlabel=r"Imaginary time $\tau$", ylabel="Position x")
ax.legend()
plt.show()

Everything outside those two links cancels when we subtract the old action from the proposed action.

## Compute the local action change

Replace the two old links by the two proposed links.

In [ ]:
@njit
def delta_action(path, x_prime, i, dt):
    left = path[(i - 1) % path.size]
    right = path[(i + 1) % path.size]
    new = action(left, x_prime, dt) + action(x_prime, right, dt)
    old = action(left, path[i], dt) + action(path[i], right, dt)
    return new - old

This works at $i=0$ and at the last stored point because both neighbor indices wrap around.

## Check the local calculation

Recompute the full action once and compare. This catches missing links and periodic-boundary mistakes.

In [ ]:
local_change = delta_action(example_path, proposal[i], i, dt)
full_change = total_action(proposal, dt) - total_action(example_path, dt)
print(f"Local change: {local_change:.8f}")
print(f"Full change:  {full_change:.8f}")
np.testing.assert_allclose(local_change, full_change, atol=1e-12)

Agreement here checks the action calculation. It does not yet show that a Markov chain has equilibrated.

## Accept or reject

For a symmetric proposal, $A=\min(1,e^{-\Delta S_E/\hbar})$. Lower-action moves are accepted; higher-action moves are sometimes accepted.

A rejection leaves the path unchanged and still counts as an attempt.

In [ ]:
action_changes = np.linspace(-3, 5, 300)
accept_probability = np.exp(-np.maximum(action_changes, 0) / HBAR)
fig, ax = plt.subplots()
ax.plot(action_changes, accept_probability, color="#007c91", lw=2)
ax.axvline(0, color="black", lw=1, ls="--")
ax.set(xlabel="Proposed action change ΔS", ylabel="Acceptance probability",
       ylim=(-0.03, 1.05))
plt.show()

The acceptance probability is about 37% for an action increase of one in these units. Rejections belong in the recorded chain; they are not discarded observations.

## One Metropolis update

The oscillator has no hard position walls. We let the path explore the real line.

In [ ]:
@njit
def update_point(path, i, dt, hitsize, rng):
    x_prime = path[i] + rng.uniform(-hitsize, hitsize)
    change = delta_action(path, x_prime, i, dt)
    if change <= 0 or rng.random() < np.exp(-change / HBAR):
        path[i] = x_prime
        return 1
    return 0

Plot limits only control the picture. Redrawing a proposal until it lies inside the plotting window would change its probability near the edges.

## One sweep

Visit every stored point once, in order. Each accepted change is immediately used by the following update.

In [ ]:
@njit
def sweep(path, dt, hitsize, rng):
    accepted = 0
    for i in range(path.size):
        accepted += update_point(path, i, dt, hitsize, rng)
    return accepted / path.size

A fixed sequential sweep is valid here. One sweep produces the next whole-path configuration; successive paths are correlated.

## Warm up, then collect measurements

Start from zeros. Keep a trace of the warm-up, but exclude it from the histogram and reported average.

In [ ]:
@njit
def sample_paths(n_tau, dt, hitsize, burn_in, sweeps, rng):
    path = np.zeros(n_tau)
    warmup = np.empty(burn_in)
    samples = np.empty((sweeps, n_tau))
    acceptance = np.empty(sweeps)
    for k in range(burn_in):
        sweep(path, dt, hitsize, rng)
        warmup[k] = np.mean(path**2)
    for k in range(sweeps):
        acceptance[k] = sweep(path, dt, hitsize, rng)
        samples[k] = path
    return warmup, samples, acceptance

Each row of `samples` is one retained path. Copying it into a row preserves that configuration before the next sweep changes `path`.

## Run the coarse time grid

Start with $\Delta\tau=1$ and total imaginary time 30. Record one whole path per sweep, then measure its mean square position.

In [ ]:
DT = 1.0
NTAU = int(round(TAU / DT))
HITSIZE = 1.0
BURN_IN = 10_000
SWEEPS = 200_000
warmup, samples, acceptance = sample_paths(
    NTAU, DT, HITSIZE, BURN_IN, SWEEPS, np.random.default_rng(142))
x2_trace = np.mean(samples**2, axis=1)
print(f"{samples.shape[0]:,} paths; {samples.shape[1]} points per path")
print(f"Accepted fraction: {acceptance.mean():.3f}")

The first call includes compilation. Rerunning this cell restarts the chain from the same seed; it does not accumulate old measurements.

## The finite grid has its own answer

For a long harmonic-oscillator path with this left-endpoint action,

$$\langle x^2\rangle_{\Delta\tau}\simeq
\frac{\hbar}{m\omega\sqrt{4+(\omega\Delta\tau)^2}}.$$

At $\Delta\tau=1$, this is **0.4472**, not 0.5. The exact finite-path result can also be found by summing its normal modes.

In [ ]:
def lattice_variance(n_tau, dt):
    modes = np.arange(n_tau)
    eigenvalues = dt * OMEGA**2 + 4 / dt * np.sin(np.pi * modes / n_tau)**2
    return HBAR / M * np.mean(1 / eigenvalues)

variance_dt = lattice_variance(NTAU, DT)
print(f"Finite-grid prediction: {variance_dt:.6f}")

This is a reference calculation, not part of the sampler. Keeping the left-endpoint potential makes the time-step error visible.

## Watch equilibration and fluctuations

For each path, measure $\overline{x^2}=N_\tau^{-1}\sum_i x_i^2$. Each plotted point is **one path measurement**, not one coordinate.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6.2, 4.0), sharey=True)
axes[0].plot(np.arange(400), warmup[:400], color="#a54632", lw=0.8)
axes[0].set(title="First 400 discarded sweeps", xlabel="Warm-up sweep")
axes[1].plot(np.arange(400), x2_trace[:400], color="#007c91", lw=0.8)
axes[1].set(title="First 400 retained sweeps", xlabel="Retained sweep")
for ax in axes:
    ax.axhline(variance_dt, color="black", ls="--", lw=1)
    ax.set(ylabel="Path mean of x²")
plt.show()

The dashed line is the finite-grid mean. Individual paths should fluctuate around it; they should not settle to a constant. A short trace alone cannot establish equilibrium.

## Does the starting path matter?

Run independent chains from all zeros and from a flat path at x=3. Watch both relax toward the same finite-grid mean.

In [ ]:
start_traces = []
for start, seed in ((0.0, 810), (3.0, 811)):
    path = np.full(NTAU, start)
    rng_start = np.random.default_rng(seed)
    trace = [np.mean(path**2)]
    for _ in range(600):
        sweep(path, DT, HITSIZE, rng_start)
        trace.append(np.mean(path**2))
    start_traces.append(trace)
fig, ax = plt.subplots()
for start, trace in zip((0, 3), start_traces):
    ax.plot(trace, lw=1, label=f"Initial x={start} everywhere")
ax.axhline(variance_dt, color="black", ls="--", label="Finite-grid mean")
ax.set(xlabel="Sweep from initialization", ylabel="Path mean of x²")
ax.legend(fontsize=10)
plt.show()

A poor start can produce a large transient. These runs motivate discarding warm-up; agreement of long runs and repeated seeds is a stronger check than appearance alone.

## Several sampled paths

The chain samples whole paths. The position histogram combines the coordinates from many retained paths.

In [ ]:
fig, ax = plt.subplots()
for k in [0, SWEEPS // 2, SWEEPS - 1]:
    closed = np.r_[samples[k], samples[k, 0]]
    ax.plot(np.arange(NTAU + 1) * DT, closed, lw=1, label=f"Path {k}")
ax.set(xlabel=r"Imaginary time $\tau$", ylabel="Position x")
ax.legend()
plt.show()

Coordinates within one path are correlated too. Counting all coordinates as independent samples would underestimate the uncertainty.

## Two different directions in the data

One row is a path. Moving across a row follows imaginary time; moving down follows Monte Carlo sweeps.

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(samples[:200], aspect="auto", origin="lower",
               extent=(-DT / 2, TAU - DT / 2, -0.5, 199.5),
               cmap="RdBu_r", vmin=-2.5, vmax=2.5)
ax.set(xlabel="Imaginary time τ within a path", ylabel="Retained Monte Carlo sweep")
fig.colorbar(im, ax=ax, label="Position x")
plt.show()

Neighboring colors persist in both directions. Nearby coordinates are correlated within a path, and successive paths are correlated along the Markov chain. This is not real-time motion.

## Compare probability densities

First compare with the distribution the finite grid actually samples. Then compare with the continuum ground state.

In [ ]:
rho_lattice = np.exp(-x_grid**2 / (2 * variance_dt)) / np.sqrt(
    2 * np.pi * variance_dt)
fig, ax = plt.subplots()
ax.hist(samples.ravel(), bins=100, density=True, color="#9fc9d3", label="MCMC")
ax.plot(x_grid, rho_lattice, color="#172b4d", lw=2, label="Finite grid")
ax.plot(x_grid, rho_ground, "--", color="#a54632", lw=2, label="Continuum")
ax.set(xlim=(-3, 3), xlabel="Position x", ylabel="Probability density")
ax.legend()
plt.show()

A narrower histogram at this time step is expected. More sweeps reduce sampling noise; they do not remove this discretization error.

## Watch the histogram accumulate

Use the same bins and axes so changes in the density come from the data.

In [ ]:
bins = np.linspace(-4.5, 4.5, 91)
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(6.2, 3.6))
for ax, count in zip(axes, (20, SWEEPS)):
    ax.hist(samples[:count].ravel(), bins=bins, density=True,
            color="#9fc9d3")
    ax.plot(x_grid, rho_lattice, color="#172b4d", lw=2)
    ax.set(title=f"{count:,} paths", xlabel="Position x", xlim=(-3, 3), ylim=(0, 0.85))
axes[0].set_ylabel("Density ρ(x)")
plt.show()

The dark curve is the exact finite-grid density. More paths make the histogram smoother. They do not make adjacent coordinates or adjacent sweeps independent.

## More samples reduce noise, not grid bias

Average the path measurements accumulated so far and compare with both known targets.

In [ ]:
counts = np.arange(1, SWEEPS + 1)
running_x2 = np.cumsum(x2_trace) / counts
fig, ax = plt.subplots()
ax.semilogx(counts, running_x2, color="#007c91", label="Running MCMC mean")
ax.axhline(variance_dt, color="#172b4d", label="Exact finite grid")
ax.axhline(0.5, color="#a54632", ls="--", label="Continuum ground state")
ax.set(xlabel="Number of retained paths", ylabel="Running <x²>")
ax.legend(fontsize=10)
plt.show()

The running mean approaches the finite-grid answer, around 0.447. Its persistent gap from 0.5 is time-step error, not a reason to keep running this same grid forever.

## Measure the memory between sweeps

For $q_s=\overline{x^2}_s$, estimate $C(k)=\operatorname{Cov}(q_s,q_{s+k})/\operatorname{Var}(q_s)$.

This is correlation in **Monte Carlo sweep number**, not in imaginary time.

In [ ]:
def autocorrelation(values, max_lag):
    centered = np.asarray(values) - np.mean(values)
    n = centered.size
    fft_size = 1 << (2 * n - 1).bit_length()
    spectrum = np.fft.rfft(centered, n=fft_size)
    covariance = np.fft.irfft(spectrum * spectrum.conj(), n=fft_size)
    covariance = covariance[:max_lag + 1] / (n - np.arange(max_lag + 1))
    return covariance / covariance[0]

Zero padding prevents the Fourier calculation from wrapping the end of the chain onto the beginning. Dividing by the number of pairs accounts for each lag.

## Nearby sweeps carry repeated information

A slowly decaying correlation means many recorded paths can still contain relatively little independent information.

In [ ]:
lags = np.arange(101)
fig, ax = plt.subplots()
ax.plot(lags, autocorrelation(x2_trace, 100), color="#007c91")
ax.axhline(0, color="black", lw=1)
ax.set(xlabel="Lag in Monte Carlo sweeps", ylabel="Correlation of path mean x²",
       ylim=(-0.15, 1.05))
plt.show()

The correlation is one at zero lag and decays over several sweeps. Treating all 200,000 path measurements as independent would give an error bar that is too small.

## Put an uncertainty on the width

Use the per-sweep measurements and the blocking method from Lab 12. Neighboring blocks must be long enough to be approximately independent.

In [ ]:
def blocked_mean_error(values, block_size):
    n_blocks = values.size // block_size
    blocks = values[:n_blocks * block_size].reshape(n_blocks, block_size)
    means = blocks.mean(axis=1)
    return values.mean(), means.std(ddof=1) / np.sqrt(n_blocks)

for block_size in (250, 500, 1000, 2000):
    mean, error = blocked_mean_error(x2_trace, block_size)
    print(f"Block {block_size:4d}: <x²> = {mean:.4f} ± {error:.4f}")

Look for an error estimate that settles as the block size grows, while retaining enough blocks. If it keeps growing, collect a longer chain.

## See the uncertainty change with block size

Average consecutive measurements into blocks. As blocks become longer than the correlation scale, the error estimate should settle.

In [ ]:
block_sizes = np.array([1, 2, 5, 10, 20, 50, 100, 250, 500, 1000, 2000])
block_errors = [blocked_mean_error(x2_trace, b)[1] for b in block_sizes]
naive_error = x2_trace.std(ddof=1) / np.sqrt(SWEEPS)
fig, ax = plt.subplots()
ax.semilogx(block_sizes, block_errors, "o-", color="#007c91", label="Blocked error")
ax.axhline(naive_error, color="#a54632", ls="--", label="Assume independent sweeps")
ax.set(xlabel="Sweeps per block", ylabel="Estimated standard error of <x²>")
ax.legend(fontsize=10)
plt.show()

Large blocks leave fewer block means: 1,000 sweeps per block gives 200 blocks; 2,000 gives 100. A noisy plateau is expected, and its height matters more than one selected block size.

## Acceptance is not the same as exploration

Compare small, moderate, and very large proposals at the same time grid and run length. All three target the same distribution.

In [ ]:
proposal_sizes = (0.1, 1.0, 8.0)
proposal_traces, proposal_acceptance = [], []
for run, h in enumerate(proposal_sizes):
    _, trial_samples, trial_acceptance = sample_paths(
        NTAU, DT, h, BURN_IN, SWEEPS, np.random.default_rng(920 + run))
    proposal_traces.append(np.mean(trial_samples**2, axis=1))
    proposal_acceptance.append(trial_acceptance.mean())
    print(f"h={h:.1f}: accepted {trial_acceptance.mean():.1%}")

Small proposals usually get accepted but move slowly. Very large proposals often get rejected. A useful proposal moves a reasonable distance and still gets accepted often enough.

## Which proposal forgets its past fastest?

Compare correlations of the same observable over the same sweep lags.

In [ ]:
fig, ax = plt.subplots()
for h, trace, accepted in zip(proposal_sizes, proposal_traces, proposal_acceptance):
    ax.plot(np.arange(501), autocorrelation(trace, 500),
            label=f"h={h:g}; accepted {accepted:.0%}")
ax.axhline(0, color="black", lw=1)
ax.set(xlabel="Lag in Monte Carlo sweeps", ylabel="Correlation of path mean x²",
       ylim=(-0.15, 1.05))
ax.legend(fontsize=10)
plt.show()

The smallest proposal has the highest acceptance and the longest memory. Here h=1 decorrelates faster; these choices are illustrative, not a universal optimum.

## Reduce the time step

Keep the total imaginary time fixed. Halving the step doubles the number of coordinates in each path.

In [ ]:
time_steps = np.array([1.0, 0.5, 0.25])
traces_by_dt = [x2_trace]
means, errors = [blocked_mean_error(x2_trace, 1000)[0]], [
    blocked_mean_error(x2_trace, 1000)[1]]
for run, smaller_dt in enumerate(time_steps[1:], start=1):
    smaller_n = int(round(TAU / smaller_dt))
    _, fine_samples, _ = sample_paths(
        smaller_n, smaller_dt, HITSIZE * np.sqrt(smaller_dt),
        BURN_IN, SWEEPS, np.random.default_rng(142 + run))
    trace = np.mean(fine_samples**2, axis=1)
    traces_by_dt.append(trace)
    mean, error = blocked_mean_error(trace, 1000)
    means.append(mean)
    errors.append(error)

Scaling the proposal with $\sqrt{\Delta\tau}$ keeps the kinetic-action change comparable. We still need to check mixing as the time grid gets finer.

## A finer time grid also changes the sampling

The kinetic term couples neighboring points more strongly as the time step shrinks. Local updates then need more sweeps to move the whole path.

In [ ]:
fig, ax = plt.subplots()
for step, trace in zip(time_steps, traces_by_dt):
    ax.plot(np.arange(301), autocorrelation(trace, 300), label=f"Δτ={step:g}")
ax.axhline(0, color="black", lw=1)
ax.set(xlabel="Lag in Monte Carlo sweeps", ylabel="Correlation of path mean x²",
       ylim=(-0.15, 1.05))
ax.legend()
plt.show()

Finer discretization reduces bias but slows this local sampler. Equal numbers of sweeps do not give equal precision, even after scaling the proposal size.

## Recheck the errors after refinement

Use the same block sizes for each chain so the change in correlation is visible.

In [ ]:
fig, ax = plt.subplots()
for step, trace in zip(time_steps, traces_by_dt):
    estimated = [blocked_mean_error(trace, b)[1] for b in block_sizes]
    ax.semilogx(block_sizes, estimated, "o-", ms=3, label=f"Δτ={step:g}")
ax.set(xlabel="Sweeps per block", ylabel="Estimated standard error of <x²>")
ax.legend()
plt.show()

The finest chain needs blocks of about 1,000–2,000 sweeps before the estimate settles. We used 200,000 retained paths per grid so 1,000-sweep blocks still leave 200 block means. The following bars use that block size.

## Separate the two errors

The error bars estimate sampling uncertainty. The curve shows the bias from the finite time step.

In [ ]:
dt_curve = np.linspace(0, 1.05, 200)
variance_curve = HBAR / (M * OMEGA * np.sqrt(4 + (OMEGA * dt_curve)**2))
fig, ax = plt.subplots()
ax.plot(dt_curve, variance_curve, color="#172b4d", label="Long-path grid prediction")
ax.axhline(0.5, color="#a54632", ls="--", label="Continuum ground state")
ax.errorbar(time_steps, means, yerr=errors, fmt="o", capsize=4,
            color="#007c91", label="MCMC ± blocked error")
ax.set(xlabel=r"Time step $\Delta\tau$", ylabel=r"$\langle x^2\rangle$")
ax.legend(fontsize=10)
plt.show()

Agreement with the grid prediction checks the sampler; convergence toward 0.5 checks the time discretization. Bars show one standard error, so individual points need not overlap the prediction.

## Put both numerical methods against the exact density

Use the finest spatial grid and the finest time grid computed above.

In [ ]:
fd_x, fd_rho, _, _, _ = fd_results[-1]
fig, ax = plt.subplots()
ax.hist(fine_samples.ravel(), bins=bins, density=True,
        color="#9fc9d3", alpha=0.8, label="MCMC Δτ=0.25")
ax.plot(x_grid, rho_ground, color="black", lw=2, label="Exact ground state")
ax.plot(fd_x, fd_rho, "--", color="#a54632", lw=2, label="Finite differences Δx=0.05")
ax.set(xlim=(-3, 3), xlabel="Position x", ylabel="Density ρ(x)")
ax.legend(fontsize=9)
plt.show()

Both methods recover the same physical distribution. The smooth deterministic curve has spatial discretization error; the MCMC histogram has time-step bias and sampling fluctuations.

## Why the tails matter to $\langle x^2\rangle$

The width is the area under $x^2\rho(x)$. Weighting by $x^2$ suppresses the central peak and emphasizes the shoulders.

In [ ]:
density, edges = np.histogram(fine_samples.ravel(), bins=bins, density=True)
centers = (edges[:-1] + edges[1:]) / 2
fig, ax = plt.subplots()
ax.step(centers, centers**2 * density, where="mid", color="#007c91", label="MCMC")
ax.plot(x_grid, x_grid**2 * rho_ground, color="black", label="Exact")
ax.plot(fd_x, fd_x**2 * fd_rho, "--", color="#a54632", label="Finite differences")
ax.set(xlim=(-3, 3), ylim=(0, 0.32), xlabel="Position x", ylabel="Contribution x²ρ(x)")
ax.legend(fontsize=9, loc="upper center")
plt.show()

Getting the center of the histogram right is not enough to get its width right. The plotted histogram uses bin centers; the reported MCMC width is computed directly from the samples.

## Compare the same scalar observable

Keep the exact answer fixed while refining each numerical method.

In [ ]:
labels = [f"FD Δx={dx:g}" for dx in fd_dx] + [f"MCMC Δτ={dt:g}" for dt in time_steps]
fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.axvline(0.5, color="black", ls="--", label="Exact ground state")
ax.plot(fd_x2, np.arange(4), "s", color="#a54632", label="Finite differences")
ax.errorbar(means, np.arange(4, 7), xerr=errors, fmt="o", capsize=4,
            color="#007c91", label="MCMC ± blocked error")
ax.set(yticks=np.arange(7), yticklabels=labels, xlabel="Mean square position <x²>")
ax.invert_yaxis()
ax.legend(fontsize=9, loc="lower left")
plt.show()

FD points have no sampling error bars; that does not mean they are exact. MCMC bars describe sampling uncertainty only. Both discretizations must be refined to approach 0.5.

## Which change addresses which discrepancy?

| Change | What it addresses |
|---|---|
| Finer spatial grid / wider spatial domain | Finite-difference truncation / boundary error |
| More retained sweeps | MCMC sampling uncertainty, once equilibrated |
| Longer discarded warm-up / different starts | Sensitivity to the initial path |
| Different proposal size | Acceptance and exploration speed |
| Smaller $\Delta\tau$ at fixed total time | Path-integral time-discretization error |
| Longer total imaginary time | Excited-state contributions |

Use the **exact solution** to measure the error and an **independent method** to check the numerical approach.

## Carry this into Assignment 3

Follow the assignment's specified total imaginary time and action. At finite total time, compare with the thermal result, not automatically with the ground state:

$$\langle x^2\rangle=\frac{\hbar}{2m\omega}\coth\!\left(\frac{\omega\tau_{\rm total}}2\right).$$

For each result, state the time step, run length, discarded warm-up, and uncertainty method. Check the sampler against the finite grid before interpreting a continuum discrepancy.

## What this comparison established

1. The harmonic oscillator supplies an exact density and $\langle x^2\rangle=0.5$.
2. Finite differences recover them by solving a deterministic eigenproblem; refinement reduces spatial bias.
3. Path-integral MCMC recovers them by sampling paths; diagnostics reveal equilibration and correlation.
4. The exact finite-grid answer separates a sampler problem from time-step bias.
5. Both methods approach the same continuum answer. An error bar alone does not account for discretization error.

Adapted from Javier Duarte's Lab 14 and the Mittal et al. example discussed in lecture. The left-endpoint potential follows the corrected course notebook.